# National Diabetes, Heat, and Residential Air-Conditioning Study

**Design:** U.S. census-tract ecological study with Tennessee and Shelby County subanalyses.

This notebook:

1. Uploads and audits the cleaned CDC PLACES–Census LACE dataset.
2. Downloads 2023 ACS 5-year socioeconomic covariates.
3. Creates tract-level heat exposure using Google Earth Engine and Daymet V4.
4. Merges all datasets.
5. Runs national, Tennessee, Shelby County, and sensitivity models.
6. Exports clean data, tables, figures, and model summaries.

> **Interpretation rule:** All findings are neighborhood-level associations. They do not establish individual-level causation.


## Before running

You need:

- The file `national_places_lace_clean.csv`
- A Census API key
- A Google Earth Engine account and Cloud project

In Colab, choose **Runtime → Run all**. Some cells ask you to enter information or upload files.


In [ ]:
# Install required packages
!pip -q install earthengine-api geemap statsmodels openpyxl pyarrow


In [ ]:
# Imports and project folders
from pathlib import Path
from getpass import getpass
import io
import json
import os
import time
import zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from google.colab import files

ROOT = Path("/content/diabetes_heat_ac_project")
RAW = ROOT / "data_raw"
CLEAN = ROOT / "data_clean"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"
MODELS = ROOT / "models"

for folder in [ROOT, RAW, CLEAN, TABLES, FIGURES, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {ROOT}")


## 1. Upload the cleaned PLACES–LACE dataset

In [ ]:
uploaded = files.upload()

expected_name = "national_places_lace_clean.csv"
if expected_name in uploaded:
    data_bytes = uploaded[expected_name]
else:
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("Upload exactly one CSV file: national_places_lace_clean.csv")
    expected_name = csv_names[0]
    data_bytes = uploaded[expected_name]

input_path = RAW / "national_places_lace_clean.csv"
input_path.write_bytes(data_bytes)

df = pd.read_csv(
    input_path,
    dtype={"GEOID": str, "CountyFIPS": str, "TractFIPS": str},
    low_memory=False
)

df["GEOID"] = df["GEOID"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(11)
print(df.shape)
df.head()


## 2. Validate columns and construct the primary cohort

The principal analysis uses the contiguous United States and Washington, D.C. Alaska, Hawaii, and Puerto Rico are reserved for sensitivity analyses because their cooling and climate patterns differ substantially.


In [ ]:
required_columns = [
    "GEOID", "StateAbbr", "StateDesc", "CountyName", "TotalPopulation",
    "DIABETES_CrudePrev", "NO_AC_PE", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev", "DISABILITY_CrudePrev"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

numeric_columns = [
    "TotalPopulation", "DIABETES_CrudePrev", "NO_AC_PE",
    "OBESITY_CrudePrev", "LPA_CrudePrev", "ACCESS2_CrudePrev",
    "DISABILITY_CrudePrev"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert impossible/sentinel values to missing.
for col in numeric_columns:
    df.loc[(df[col] < 0), col] = np.nan

primary = df[~df["StateAbbr"].isin(["AK", "HI", "PR"])].copy()
primary = primary.dropna(subset=[
    "DIABETES_CrudePrev", "NO_AC_PE", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev"
])

print(f"All rows: {len(df):,}")
print(f"Primary cohort rows: {len(primary):,}")
print(f"Duplicate GEOIDs: {primary['GEOID'].duplicated().sum():,}")
print(primary["GEOID"].str.len().value_counts().sort_index())


In [ ]:
# Data-quality table
audit_rows = []
for col in numeric_columns:
    audit_rows.append({
        "variable": col,
        "n": primary[col].notna().sum(),
        "missing_n": primary[col].isna().sum(),
        "missing_percent": primary[col].isna().mean() * 100,
        "minimum": primary[col].min(),
        "median": primary[col].median(),
        "maximum": primary[col].max()
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(TABLES / "data_quality_audit.csv", index=False)
audit


## 3. Download 2023 ACS 5-year tract covariates

Enter your Census API key when prompted. The key is held only in the current Colab session and is not written to the output files.


In [ ]:
CENSUS_API_KEY = getpass("Paste your Census API key, or press Enter to continue without one: ").strip()
if CENSUS_API_KEY:
    print("Census API key detected.")
else:
    print("No Census API key supplied. The notebook will continue with unauthenticated requests, which may be slower or rate-limited.")

ACS_YEAR = 2023
ACS_BASE = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

age65_vars = [
    "B01001_020E", "B01001_021E", "B01001_022E",
    "B01001_023E", "B01001_024E", "B01001_025E",
    "B01001_044E", "B01001_045E", "B01001_046E",
    "B01001_047E", "B01001_048E", "B01001_049E"
]

acs_vars = [
    "NAME",
    "B01003_001E",  # total population
    "B17001_001E",  # poverty universe
    "B17001_002E",  # below poverty
    "B19013_001E",  # median household income
    "B25003_001E",  # occupied housing units
    "B25003_003E",  # renter occupied
    "B08201_001E",  # households
    "B08201_002E",  # no vehicle
] + age65_vars

state_fips_values = sorted(primary["GEOID"].str[:2].dropna().unique())
len(state_fips_values), state_fips_values[:5]


In [ ]:
def fetch_acs_state(state_fips, max_retries=4):
    params = {
        "get": ",".join(acs_vars),
        "for": "tract:*",
        "in": f"state:{state_fips} county:*"
    }
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    last_error = None
    for attempt in range(max_retries):
        try:
            response = requests.get(ACS_BASE, params=params, timeout=120)
            response.raise_for_status()
            payload = response.json()
            return pd.DataFrame(payload[1:], columns=payload[0])
        except Exception as exc:
            last_error = exc
            time.sleep(2 ** attempt)

    raise RuntimeError(f"ACS download failed for state {state_fips}: {last_error}")

acs_parts = []
for i, state_fips in enumerate(state_fips_values, start=1):
    part = fetch_acs_state(state_fips)
    acs_parts.append(part)
    print(f"[{i}/{len(state_fips_values)}] downloaded state FIPS {state_fips}: {len(part):,} tracts")

acs = pd.concat(acs_parts, ignore_index=True)
acs.to_csv(RAW / "acs_2023_raw.csv", index=False)
print(acs.shape)


In [ ]:
# Clean ACS fields and create tract GEOIDs.
acs["GEOID"] = (
    acs["state"].astype(str).str.zfill(2)
    + acs["county"].astype(str).str.zfill(3)
    + acs["tract"].astype(str).str.zfill(6)
)

for col in [v for v in acs_vars if v != "NAME"]:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")
    acs.loc[acs[col] < 0, col] = np.nan

acs["age65_number"] = acs[age65_vars].sum(axis=1, min_count=len(age65_vars))

def safe_percent(numerator, denominator):
    result = numerator / denominator * 100
    return result.where(denominator > 0)

acs["poverty_percent"] = safe_percent(acs["B17001_002E"], acs["B17001_001E"])
acs["age65_percent"] = safe_percent(acs["age65_number"], acs["B01003_001E"])
acs["renter_percent"] = safe_percent(acs["B25003_003E"], acs["B25003_001E"])
acs["no_vehicle_percent"] = safe_percent(acs["B08201_002E"], acs["B08201_001E"])
acs["median_household_income"] = acs["B19013_001E"]
acs["acs_total_population"] = acs["B01003_001E"]

acs_keep = acs[[
    "GEOID", "NAME", "acs_total_population", "poverty_percent",
    "age65_percent", "renter_percent", "no_vehicle_percent",
    "median_household_income"
]].copy()

acs_keep.to_csv(CLEAN / "acs_2023_clean.csv", index=False)

analysis = primary.merge(acs_keep, on="GEOID", how="left", validate="one_to_one")
print(f"ACS match rate: {analysis['poverty_percent'].notna().mean():.2%}")
analysis.head()


## 4. Generate heat exposure with Google Earth Engine

This cell creates 2019–2023 Daymet heat metrics for 2020 Census tracts:

- Mean annual days with maximum temperature ≥90°F
- Mean annual days with maximum temperature ≥95°F
- Mean June–August maximum temperature

The Earth Engine export runs as a task and writes a CSV to your Google Drive. After it finishes, download that CSV and upload it in the next section.

**Note:** Daymet `tmax` is measured in degrees Celsius.


In [ ]:
import ee

EE_PROJECT = input("Enter your Google Cloud / Earth Engine project ID: ").strip()
if not EE_PROJECT:
    raise ValueError("Enter the project ID connected to your Earth Engine account.")

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")


In [ ]:
# Build annual Daymet heat images, then average 2019–2023.
tracts = ee.FeatureCollection("TIGER/2020/TRACT").filter(
    ee.Filter.inList("STATEFP", state_fips_values)
)

daymet = ee.ImageCollection("NASA/ORNL/DAYMET_V4").select("tmax")

def annual_heat_image(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    summer_start = ee.Date.fromYMD(year, 6, 1)
    summer_end = ee.Date.fromYMD(year, 9, 1)

    annual = daymet.filterDate(start, end)
    summer = daymet.filterDate(summer_start, summer_end)

    hot90 = annual.map(
        lambda image: image.gte(32.2222).rename("hot90")
    ).sum().rename("hot_days_90f")

    hot95 = annual.map(
        lambda image: image.gte(35.0).rename("hot95")
    ).sum().rename("hot_days_95f")

    summer_tmax = summer.mean().rename("summer_tmax_c")

    return hot90.addBands(hot95).addBands(summer_tmax).set("year", year)

annual_images = ee.ImageCollection.fromImages([
    annual_heat_image(year) for year in range(2019, 2024)
])

heat_mean = annual_images.mean().rename([
    "hot_days_90f", "hot_days_95f", "summer_tmax_c"
])

# Reduce the five-year mean image over every tract.
tract_heat = heat_mean.reduceRegions(
    collection=tracts,
    reducer=ee.Reducer.mean(),
    scale=1000,
    tileScale=8
).map(lambda feature: ee.Feature(None, {
    "GEOID": feature.get("GEOID"),
    "STATEFP": feature.get("STATEFP"),
    "COUNTYFP": feature.get("COUNTYFP"),
    "TRACTCE": feature.get("TRACTCE"),
    "hot_days_90f": feature.get("hot_days_90f"),
    "hot_days_95f": feature.get("hot_days_95f"),
    "summer_tmax_c": feature.get("summer_tmax_c")
}))

task = ee.batch.Export.table.toDrive(
    collection=tract_heat,
    description="tract_heat_daymet_2019_2023",
    folder="EarthEngineExports",
    fileNamePrefix="tract_heat_daymet_2019_2023",
    fileFormat="CSV"
)
task.start()

print("Earth Engine task started.")
print("Task status:", task.status())
print("Open the Earth Engine Tasks tab or Google Drive/EarthEngineExports to retrieve the CSV after completion.")


## 5. Upload the completed Earth Engine heat CSV

Run this section after the Earth Engine export has finished.


In [ ]:
uploaded_heat = files.upload()
heat_csvs = [name for name in uploaded_heat if name.lower().endswith(".csv")]
if len(heat_csvs) != 1:
    raise ValueError("Upload exactly one Earth Engine heat CSV.")

heat_name = heat_csvs[0]
heat_path = RAW / "tract_heat_daymet_2019_2023.csv"
heat_path.write_bytes(uploaded_heat[heat_name])

heat = pd.read_csv(heat_path, dtype={"GEOID": str}, low_memory=False)
heat["GEOID"] = heat["GEOID"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(11)

for col in ["hot_days_90f", "hot_days_95f", "summer_tmax_c"]:
    heat[col] = pd.to_numeric(heat[col], errors="coerce")

heat = heat.drop_duplicates(subset=["GEOID"])
analysis = analysis.drop(columns=[
    c for c in ["hot_days_90f", "hot_days_95f", "summer_tmax_c"] if c in analysis.columns
])
analysis = analysis.merge(
    heat[["GEOID", "hot_days_90f", "hot_days_95f", "summer_tmax_c"]],
    on="GEOID",
    how="left",
    validate="one_to_one"
)

print(f"Heat match rate: {analysis['hot_days_90f'].notna().mean():.2%}")
analysis.head()


## 6. Final cleaning and descriptive statistics

In [ ]:
analysis_variables = [
    "DIABETES_CrudePrev", "NO_AC_PE", "hot_days_90f", "hot_days_95f",
    "summer_tmax_c", "poverty_percent", "age65_percent", "renter_percent",
    "no_vehicle_percent", "median_household_income", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev", "DISABILITY_CrudePrev",
    "TotalPopulation"
]

for col in analysis_variables:
    analysis[col] = pd.to_numeric(analysis[col], errors="coerce")
    analysis.loc[np.isinf(analysis[col]), col] = np.nan

analysis.to_csv(CLEAN / "national_analysis_merged.csv", index=False)

summary = analysis[analysis_variables].describe(
    percentiles=[0.25, 0.5, 0.75]
).T
summary["missing_n"] = analysis[analysis_variables].isna().sum()
summary["missing_percent"] = analysis[analysis_variables].isna().mean() * 100
summary.to_csv(TABLES / "table1_descriptive_statistics.csv")
summary


In [ ]:
# Unadjusted correlations
corr_data = analysis[["DIABETES_CrudePrev", "NO_AC_PE", "hot_days_90f"]].dropna()

rho_ac, p_ac = spearmanr(corr_data["DIABETES_CrudePrev"], corr_data["NO_AC_PE"])
rho_heat, p_heat = spearmanr(corr_data["DIABETES_CrudePrev"], corr_data["hot_days_90f"])

correlations = pd.DataFrame([
    {"comparison": "Diabetes vs no AC", "spearman_rho": rho_ac, "p_value": p_ac},
    {"comparison": "Diabetes vs hot days", "spearman_rho": rho_heat, "p_value": p_heat}
])
correlations.to_csv(TABLES / "unadjusted_correlations.csv", index=False)
correlations


## 7. Create centered exposures and interaction terms

In [ ]:
for variable in ["NO_AC_PE", "hot_days_90f", "hot_days_95f"]:
    analysis[f"{variable}_centered"] = analysis[variable] - analysis[variable].mean()

analysis["heat90_ac_interaction"] = (
    analysis["NO_AC_PE_centered"] * analysis["hot_days_90f_centered"]
)
analysis["heat95_ac_interaction"] = (
    analysis["NO_AC_PE_centered"] * analysis["hot_days_95f_centered"]
)

analysis.to_csv(CLEAN / "national_analysis_ready.csv", index=False)


## 8. Fit national regression models

Model 4 includes state fixed effects and is the principal model. HC3 robust standard errors are used.


In [ ]:
model_formulas = {
    "model1_exposures": '''
        DIABETES_CrudePrev ~
        NO_AC_PE_centered + hot_days_90f_centered + heat90_ac_interaction
    ''',
    "model2_socioeconomic": '''
        DIABETES_CrudePrev ~
        NO_AC_PE_centered + hot_days_90f_centered + heat90_ac_interaction +
        poverty_percent + age65_percent + renter_percent + no_vehicle_percent
    ''',
    "model3_full": '''
        DIABETES_CrudePrev ~
        NO_AC_PE_centered + hot_days_90f_centered + heat90_ac_interaction +
        poverty_percent + age65_percent + renter_percent + no_vehicle_percent +
        OBESITY_CrudePrev + LPA_CrudePrev + ACCESS2_CrudePrev
    ''',
    "model4_state_fixed_effects": '''
        DIABETES_CrudePrev ~
        NO_AC_PE_centered + hot_days_90f_centered + heat90_ac_interaction +
        poverty_percent + age65_percent + renter_percent + no_vehicle_percent +
        OBESITY_CrudePrev + LPA_CrudePrev + ACCESS2_CrudePrev +
        C(StateAbbr)
    '''
}

models = {}
for name, formula in model_formulas.items():
    fitted = smf.ols(formula, data=analysis).fit(cov_type="HC3")
    models[name] = fitted
    (MODELS / f"{name}.txt").write_text(fitted.summary().as_text())
    print(name, "N =", int(fitted.nobs), "R2 =", round(fitted.rsquared, 4))


In [ ]:
def tidy_model(model, model_name):
    ci = model.conf_int()
    return pd.DataFrame({
        "model": model_name,
        "term": model.params.index,
        "estimate": model.params.values,
        "std_error": model.bse.values,
        "p_value": model.pvalues.values,
        "ci_lower": ci[0].values,
        "ci_upper": ci[1].values,
        "n": model.nobs,
        "r_squared": model.rsquared,
        "adjusted_r_squared": model.rsquared_adj
    })

regression_table = pd.concat(
    [tidy_model(model, name) for name, model in models.items()],
    ignore_index=True
)
regression_table.to_csv(TABLES / "national_regression_results.csv", index=False)

key_terms = [
    "NO_AC_PE_centered", "hot_days_90f_centered", "heat90_ac_interaction",
    "poverty_percent", "age65_percent", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev"
]
regression_table[regression_table["term"].isin(key_terms)]


## 9. Tennessee and Shelby County subanalyses

In [ ]:
tennessee = analysis[analysis["StateAbbr"] == "TN"].copy()
shelby = analysis[
    (analysis["StateAbbr"] == "TN") &
    (analysis["CountyName"].str.contains("Shelby", case=False, na=False))
].copy()

sub_formula = '''
    DIABETES_CrudePrev ~
    NO_AC_PE_centered + hot_days_90f_centered + heat90_ac_interaction +
    poverty_percent + age65_percent +
    OBESITY_CrudePrev + LPA_CrudePrev + ACCESS2_CrudePrev
'''

tn_model = smf.ols(sub_formula, data=tennessee).fit(cov_type="HC3")

# A simpler model is safer for Shelby County's smaller sample.
shelby_formula = '''
    DIABETES_CrudePrev ~
    NO_AC_PE + poverty_percent + age65_percent +
    OBESITY_CrudePrev + LPA_CrudePrev
'''
shelby_model = smf.ols(shelby_formula, data=shelby).fit(cov_type="HC3")

(MODELS / "tennessee_model.txt").write_text(tn_model.summary().as_text())
(MODELS / "shelby_county_model.txt").write_text(shelby_model.summary().as_text())

subanalysis_results = pd.concat([
    tidy_model(tn_model, "Tennessee"),
    tidy_model(shelby_model, "Shelby County")
], ignore_index=True)
subanalysis_results.to_csv(TABLES / "tennessee_shelby_models.csv", index=False)

print("Tennessee tracts:", len(tennessee), "model N:", int(tn_model.nobs))
print("Shelby County tracts:", len(shelby), "model N:", int(shelby_model.nobs))


## 10. Sensitivity analyses

In [ ]:
# 95°F threshold
sensitivity_95 = smf.ols(
    '''
    DIABETES_CrudePrev ~
    NO_AC_PE_centered + hot_days_95f_centered + heat95_ac_interaction +
    poverty_percent + age65_percent + renter_percent + no_vehicle_percent +
    OBESITY_CrudePrev + LPA_CrudePrev + ACCESS2_CrudePrev +
    C(StateAbbr)
    ''',
    data=analysis
).fit(cov_type="HC3")

# Exclude low-population tracts
population_restricted = analysis[analysis["TotalPopulation"] >= 500].copy()
sensitivity_population = smf.ols(
    model_formulas["model4_state_fixed_effects"],
    data=population_restricted
).fit(cov_type="HC3")

# Population-weighted model
weighted_data = analysis.dropna(subset=[
    "TotalPopulation", "DIABETES_CrudePrev", "NO_AC_PE_centered",
    "hot_days_90f_centered", "heat90_ac_interaction", "poverty_percent",
    "age65_percent", "renter_percent", "no_vehicle_percent",
    "OBESITY_CrudePrev", "LPA_CrudePrev", "ACCESS2_CrudePrev", "StateAbbr"
]).copy()
weighted_data = weighted_data[weighted_data["TotalPopulation"] > 0]

sensitivity_weighted = smf.wls(
    model_formulas["model4_state_fixed_effects"],
    data=weighted_data,
    weights=weighted_data["TotalPopulation"]
).fit(cov_type="HC3")

sensitivity_models = {
    "95F_threshold": sensitivity_95,
    "population_500_plus": sensitivity_population,
    "population_weighted": sensitivity_weighted
}

sensitivity_table = pd.concat([
    tidy_model(model, name) for name, model in sensitivity_models.items()
], ignore_index=True)
sensitivity_table.to_csv(TABLES / "sensitivity_analysis_results.csv", index=False)

for name, model in sensitivity_models.items():
    (MODELS / f"sensitivity_{name}.txt").write_text(model.summary().as_text())
    print(name, "N =", int(model.nobs), "R2 =", round(model.rsquared, 4))


## 11. Multicollinearity assessment

In [ ]:
vif_columns = [
    "NO_AC_PE_centered", "hot_days_90f_centered", "heat90_ac_interaction",
    "poverty_percent", "age65_percent", "renter_percent",
    "no_vehicle_percent", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev"
]

vif_data = analysis[vif_columns].dropna()
X = sm.add_constant(vif_data)

vif_table = pd.DataFrame({
    "variable": X.columns,
    "VIF": [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
})

vif_table.to_csv(TABLES / "vif_table.csv", index=False)
vif_table


## 12. Figures

In [ ]:
# Figure 1: No AC and diabetes
plot_data = analysis[["NO_AC_PE", "DIABETES_CrudePrev"]].dropna()

plt.figure(figsize=(7, 5))
plt.scatter(plot_data["NO_AC_PE"], plot_data["DIABETES_CrudePrev"], alpha=0.25, s=10)
plt.xlabel("Occupied households without air conditioning (%)")
plt.ylabel("Diagnosed diabetes prevalence (%)")
plt.title("Residential Air-Conditioning Access and Diabetes Prevalence")
plt.tight_layout()
plt.savefig(FIGURES / "figure1_no_ac_diabetes.png", dpi=300)
plt.show()


In [ ]:
# Figure 2: Heat exposure and diabetes
plot_data = analysis[["hot_days_90f", "DIABETES_CrudePrev"]].dropna()

plt.figure(figsize=(7, 5))
plt.scatter(plot_data["hot_days_90f"], plot_data["DIABETES_CrudePrev"], alpha=0.25, s=10)
plt.xlabel("Mean annual days with Tmax ≥90°F, 2019–2023")
plt.ylabel("Diagnosed diabetes prevalence (%)")
plt.title("Extreme-Heat Exposure and Diabetes Prevalence")
plt.tight_layout()
plt.savefig(FIGURES / "figure2_heat_diabetes.png", dpi=300)
plt.show()


In [ ]:
# Figure 3: Predicted interaction from the principal model
principal = models["model4_state_fixed_effects"]
complete = analysis.dropna(subset=[
    "NO_AC_PE", "hot_days_90f", "poverty_percent", "age65_percent",
    "renter_percent", "no_vehicle_percent", "OBESITY_CrudePrev",
    "LPA_CrudePrev", "ACCESS2_CrudePrev", "StateAbbr"
]).copy()

heat_levels = complete["hot_days_90f"].quantile([0.25, 0.50, 0.75]).to_dict()
no_ac_grid = np.linspace(
    complete["NO_AC_PE"].quantile(0.02),
    complete["NO_AC_PE"].quantile(0.98),
    100
)

reference_state = complete["StateAbbr"].mode().iloc[0]
prediction_frames = []

for quantile, heat_value in heat_levels.items():
    frame = pd.DataFrame({
        "NO_AC_PE": no_ac_grid,
        "hot_days_90f": heat_value,
        "poverty_percent": complete["poverty_percent"].median(),
        "age65_percent": complete["age65_percent"].median(),
        "renter_percent": complete["renter_percent"].median(),
        "no_vehicle_percent": complete["no_vehicle_percent"].median(),
        "OBESITY_CrudePrev": complete["OBESITY_CrudePrev"].median(),
        "LPA_CrudePrev": complete["LPA_CrudePrev"].median(),
        "ACCESS2_CrudePrev": complete["ACCESS2_CrudePrev"].median(),
        "StateAbbr": reference_state
    })
    frame["NO_AC_PE_centered"] = frame["NO_AC_PE"] - analysis["NO_AC_PE"].mean()
    frame["hot_days_90f_centered"] = frame["hot_days_90f"] - analysis["hot_days_90f"].mean()
    frame["heat90_ac_interaction"] = (
        frame["NO_AC_PE_centered"] * frame["hot_days_90f_centered"]
    )
    frame["predicted_diabetes"] = principal.predict(frame)
    frame["heat_group"] = f"{int(quantile * 100)}th percentile heat"
    prediction_frames.append(frame)

predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(TABLES / "interaction_plot_predictions.csv", index=False)

plt.figure(figsize=(7, 5))
for label, group in predictions.groupby("heat_group"):
    plt.plot(group["NO_AC_PE"], group["predicted_diabetes"], label=label)

plt.xlabel("Occupied households without air conditioning (%)")
plt.ylabel("Predicted diagnosed diabetes prevalence (%)")
plt.title("Predicted Heat–Air-Conditioning Interaction")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "figure3_heat_ac_interaction.png", dpi=300)
plt.show()


## 13. Create priority index for mapping and planning

This index is descriptive, not causal. It averages standardized diabetes, heat, lack of AC, and poverty measures.


In [ ]:
index_vars = [
    "DIABETES_CrudePrev", "NO_AC_PE", "hot_days_90f", "poverty_percent"
]

index_data = analysis.dropna(subset=index_vars).copy()

for col in index_vars:
    index_data[f"z_{col}"] = (
        index_data[col] - index_data[col].mean()
    ) / index_data[col].std(ddof=0)

index_data["priority_index"] = index_data[
    [f"z_{col}" for col in index_vars]
].mean(axis=1)

index_data["priority_quartile"] = pd.qcut(
    index_data["priority_index"],
    q=4,
    labels=["Low", "Moderate", "High", "Critical"]
)

top_100 = index_data.sort_values("priority_index", ascending=False).head(100)
top_100.to_csv(TABLES / "top_100_priority_tracts.csv", index=False)

top_100[[
    "GEOID", "StateAbbr", "CountyName", "DIABETES_CrudePrev",
    "NO_AC_PE", "hot_days_90f", "poverty_percent", "priority_index"
]].head(20)


## 14. Export the complete project package

In [ ]:
# Create an Excel workbook containing core tables.
excel_path = ROOT / "analysis_tables.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    audit.to_excel(writer, sheet_name="Data quality", index=False)
    summary.reset_index(names="variable").to_excel(
        writer, sheet_name="Descriptive statistics", index=False
    )
    correlations.to_excel(writer, sheet_name="Correlations", index=False)
    regression_table.to_excel(writer, sheet_name="National models", index=False)
    subanalysis_results.to_excel(writer, sheet_name="TN and Shelby", index=False)
    sensitivity_table.to_excel(writer, sheet_name="Sensitivity", index=False)
    vif_table.to_excel(writer, sheet_name="VIF", index=False)
    top_100.to_excel(writer, sheet_name="Top priority tracts", index=False)

# Record a compact methods/settings file.
settings = {
    "study_design": "National census-tract ecological study",
    "primary_geography": "Contiguous United States and District of Columbia",
    "health_source": "CDC PLACES 2025 release",
    "cooling_source": "2023 Census Local Air Conditioning Estimates",
    "socioeconomic_source": "2023 ACS 5-year estimates",
    "heat_source": "Daymet V4",
    "heat_period": "2019-2023",
    "primary_heat_metric": "Mean annual days with Tmax >= 90F",
    "principal_model": "OLS with HC3 robust SE and state fixed effects",
    "interpretation": "Ecological association; no individual causal inference"
}
(ROOT / "analysis_settings.json").write_text(json.dumps(settings, indent=2))

zip_path = Path("/content/Diabetes_Heat_AC_Project_Outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT.parent))

print(f"Created: {zip_path}")
files.download(str(zip_path))


## Results interpretation checklist

Before writing the manuscript, answer these questions:

1. Is the `heat90_ac_interaction` coefficient positive or negative?
2. Is its 95% confidence interval compatible with no association?
3. Does the coefficient change after state fixed effects?
4. Are the results similar using the 95°F threshold?
5. Are the results robust after excluding low-population tracts?
6. Does population weighting materially change the conclusion?
7. Does Moran's I later show spatial autocorrelation in model residuals?

Do not interpret a statistically significant coefficient as proof that lack of AC causes diabetes. The project identifies geographic overlap and adjusted ecological associations.
